<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/03bbbbb_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
DATA_VERSION = 'PubMed_20k_RCT'
PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
DATA_DIR = f'{PROJECT_DIR}/prepared_data'
MODELS_DIR = f'{PROJECT_DIR}/models'
PREDICTIONS_DIR = f'{PROJECT_DIR}/predictions'
RESULTS_DIR = f'{PROJECT_DIR}/results'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']
SEED = 42
N_BOOTSTRAP = 2000
EXPECTED_SIZES = {'train': 180040, 'dev': 30212}

RUN_PREFIX = 'logreg'
C_CANDIDATES = [0.25, 1, 4]
MAX_ITER = 3000
print('Runs:', f'{RUN_PREFIX}_text', 'and', f'{RUN_PREFIX}_context', '| C tried:', C_CANDIDATES)

Runs: logreg_text and logreg_context | C tried: [0.25, 1, 4]


In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
for folder in (MODELS_DIR, PREDICTIONS_DIR, RESULTS_DIR):
    os.makedirs(folder, exist_ok=True)

def load_split(split):
    return pd.read_csv(f'{DATA_DIR}/{DATA_VERSION}_{split}.csv',
                       dtype={'abstract_id': str}, keep_default_na=False)

train, dev = load_split('train'), load_split('dev')
for name, df in (('train', train), ('dev', dev)):
    assert len(df) == EXPECTED_SIZES[name], f'Unexpected {name} size: re-run 02_data_preparation'
    assert set(df.label) <= set(CLASSES), f'Unknown labels in {name}'
assert not set(train.abstract_id) & set(dev.abstract_id), 'Leakage: abstract in train and dev'
print(f'Train: {train.abstract_id.nunique():,} abstracts / {len(train):,} sentences | '
      f'Dev: {dev.abstract_id.nunique():,} abstracts / {len(dev):,} sentences')

Mounted at /content/drive
Train: 15,000 abstracts / 180,040 sentences | Dev: 2,500 abstracts / 30,212 sentences


In [3]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

def bootstrap_macro_f1(y_true, y_pred, abstract_ids, n_resamples=N_BOOTSTRAP, seed=SEED):
    index, unique_ids = pd.factorize(pd.Series(abstract_ids))
    k = len(CLASSES)
    counts = np.zeros((len(unique_ids), k, k), dtype=np.int64)
    np.add.at(counts, (index, y_true, y_pred), 1)
    rng = np.random.default_rng(seed)
    scores = np.empty(n_resamples)
    for b in range(n_resamples):
        cm = counts[rng.integers(0, len(unique_ids), len(unique_ids))].sum(axis=0)
        denominator = cm.sum(axis=0) + cm.sum(axis=1)
        scores[b] = np.divide(2 * np.diag(cm), denominator, out=np.zeros(k), where=denominator > 0).mean()
    return np.percentile(scores, [2.5, 97.5])

def evaluate_and_save(run_name, model_description, input_description, df, probs,
                      fit_seconds, hardware, settings):
    probs = np.asarray(probs, dtype=float)
    assert probs.shape == (len(df), len(CLASSES)), 'probs must be (n_sentences, 5) in CLASSES order'
    k = len(CLASSES)
    y_true = pd.Categorical(df.label, categories=CLASSES).codes
    y_pred = probs.argmax(axis=1)
    one_hot = np.eye(k)[y_true]

    cm = confusion_matrix(y_true, y_pred, labels=range(k))
    tp = np.diag(cm); fn = cm.sum(axis=1) - tp; fp = cm.sum(axis=0) - tp; tn = cm.sum() - tp - fn - fp
    per_class = pd.DataFrame({
        'precision': precision_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'sensitivity': tp / (tp + fn),
        'specificity': tn / (tn + fp),
        'f1': f1_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'auroc': [roc_auc_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'auprc': [average_precision_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'prevalence': one_hot.mean(axis=0),
    }, index=pd.Index(CLASSES, name='class'))
    per_class.loc['macro average'] = per_class.mean()

    lower, upper = bootstrap_macro_f1(y_true, y_pred, df.abstract_id)
    summary = {
        'run': run_name, 'model': model_description, 'input': input_description,
        'split': 'dev', 'abstracts': df.abstract_id.nunique(), 'sentences': len(df),
        'decision_rule': 'argmax',
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=range(k), average='macro', zero_division=0),
        'macro_f1_ci95_lower': lower, 'macro_f1_ci95_upper': upper,
        'weighted_f1': f1_score(y_true, y_pred, labels=range(k), average='weighted', zero_division=0),
        'macro_auroc': per_class.loc['macro average', 'auroc'],
        'macro_auprc': per_class.loc['macro average', 'auprc'],
        'macro_sensitivity': per_class.loc['macro average', 'sensitivity'],
        'macro_specificity': per_class.loc['macro average', 'specificity'],
        'fit_seconds': round(float(fit_seconds), 1), 'hardware': hardware,
        'seed': SEED, 'n_bootstrap': N_BOOTSTRAP, 'settings': json.dumps(settings),
    }

    predictions = df[['abstract_id', 'sentence_idx', 'label']].copy()
    predictions['prediction'] = np.array(CLASSES)[y_pred]
    for i, c in enumerate(CLASSES):
        predictions[f'prob_{c}'] = probs[:, i]
    predictions.to_csv(f'{PREDICTIONS_DIR}/dev_{run_name}.csv', index=False)
    pd.DataFrame([summary]).to_csv(f'{RESULTS_DIR}/dev_{run_name}_summary.csv', index=False)
    per_class.to_csv(f'{RESULTS_DIR}/dev_{run_name}_per_class.csv')
    confusion = pd.DataFrame(cm, index=pd.Index(CLASSES, name='actual'),
                             columns=pd.Index(CLASSES, name='predicted'))
    confusion.to_csv(f'{RESULTS_DIR}/dev_{run_name}_confusion_matrix.csv')

    print(f"{model_description} | {input_description}")
    print(f"Dev macro-F1 {summary['macro_f1']:.4f} (95% CI {lower:.4f}-{upper:.4f}) | "
          f"accuracy {summary['accuracy']:.4f} | weighted F1 {summary['weighted_f1']:.4f} | "
          f"macro AUROC {summary['macro_auroc']:.4f} | macro AUPRC {summary['macro_auprc']:.4f}")
    print(f'Saved predictions and 3 result tables for run "{run_name}"')
    print('\nPer-class metrics (one class vs. the other four; decision rule: argmax):')
    display(per_class.round(4))
    print('\nConfusion matrix (rows = actual class, columns = predicted class):')
    display(confusion)
    never_predicted = [c for i, c in enumerate(CLASSES) if (y_pred == i).sum() == 0]
    print('Classes never predicted:', never_predicted or 'none')
    return summary, per_class

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import joblib

for df in (train, dev):
    df['idx_capped'] = df.sentence_idx.clip(upper=15)
    df['from_end'] = (df.n_sentences - 1 - df.sentence_idx).clip(upper=15)

def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)

def make_model(version, C):
    parts = [('sentence', tfidf(), 'text')]
    if version == 'context':
        parts += [('previous', tfidf(), 'previous'),
                  ('next', tfidf(), 'next'),
                  ('position', OneHotEncoder(handle_unknown='ignore'),
                   ['position_bin', 'idx_capped', 'from_end'])]
    return make_pipeline(ColumnTransformer(parts), LogisticRegression(C=C, max_iter=MAX_ITER))

def train_and_select(version):
    rows, best = [], None
    for C in C_CANDIDATES:
        start = time.time()
        model = make_model(version, C).fit(train, train.label)
        seconds = time.time() - start
        probs = model.predict_proba(dev)
        probs = probs[:, [list(model.classes_).index(c) for c in CLASSES]]
        score = f1_score(dev.label, np.array(CLASSES)[probs.argmax(axis=1)],
                         labels=CLASSES, average='macro', zero_division=0)
        rows.append({'C': C, 'dev_macro_f1': round(score, 4), 'fit_seconds': round(seconds, 1)})
        print(f'{version:8s} C={C:<5} dev macro-F1 {score:.4f} ({seconds:.0f} s)')
        if best is None or score > best['score']:
            best = {'C': C, 'score': score, 'model': model, 'probs': probs, 'seconds': seconds}
    joblib.dump(best['model'], f"{MODELS_DIR}/{RUN_PREFIX}_{version}.joblib")
    print(f"Chosen on the validation set: C={best['C']}")
    return best, rows

SETTINGS = {'ngram_range': [1, 2], 'min_df': 2, 'sublinear_tf': True, 'penalty': 'l2',
            'max_iter': MAX_ITER, 'C_tried': C_CANDIDATES}
HARDWARE = f'CPU ({os.cpu_count()} cores)'

In [5]:
best_text, selection_text = train_and_select('text')
summary_text, per_class_text = evaluate_and_save(
    f'{RUN_PREFIX}_text', 'TF-IDF logistic regression (L2)', 'sentence only',
    dev, best_text['probs'], best_text['seconds'], HARDWARE, {**SETTINGS, 'C': best_text['C']})

text     C=0.25  dev macro-F1 0.7492 (68 s)
text     C=1     dev macro-F1 0.7691 (90 s)
text     C=4     dev macro-F1 0.7702 (153 s)
Chosen on the validation set: C=4
TF-IDF logistic regression (L2) | sentence only
Dev macro-F1 0.7702 (95% CI 0.7634-0.7770) | accuracy 0.8334 | weighted F1 0.8313 | macro AUROC 0.9633 | macro AUPRC 0.8307
Saved predictions and 3 result tables for run "logreg_text"

Per-class metrics (one class vs. the other four; decision rule: argmax):


,precision,sensitivity,specificity,f1,auroc,auprc,prevalence
class,,,,,,,
BACKGROUND,0.6558,0.6614,0.9553,0.6586,0.9507,0.7029,0.1142
OBJECTIVE,0.6942,0.5627,0.9788,0.6216,0.9524,0.6972,0.0786
METHODS,0.8882,0.9278,0.9425,0.9076,0.9781,0.9521,0.3298
RESULTS,0.8936,0.8945,0.9486,0.8941,0.9736,0.9530,0.3257
CONCLUSIONS,0.7722,0.7665,0.9596,0.7693,0.9616,0.8481,0.1517
macro average,0.7808,0.7626,0.9570,0.7702,0.9633,0.8307,0.2000



Confusion matrix (rows = actual class, columns = predicted class):


predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
actual,,,,,
BACKGROUND,2281,442,250,34,442
OBJECTIVE,706,1337,146,24,163
METHODS,112,84,9245,449,74
RESULTS,30,7,644,8803,357
CONCLUSIONS,349,56,124,541,3512


Classes never predicted: none


In [6]:
best_context, selection_context = train_and_select('context')
summary_context, per_class_context = evaluate_and_save(
    f'{RUN_PREFIX}_context', 'TF-IDF logistic regression (L2)', 'sentence + position + neighbours',
    dev, best_context['probs'], best_context['seconds'], HARDWARE, {**SETTINGS, 'C': best_context['C']})

context  C=0.25  dev macro-F1 0.8449 (219 s)
context  C=1     dev macro-F1 0.8518 (313 s)
context  C=4     dev macro-F1 0.8519 (367 s)
Chosen on the validation set: C=4
TF-IDF logistic regression (L2) | sentence + position + neighbours
Dev macro-F1 0.8519 (95% CI 0.8454-0.8583) | accuracy 0.9010 | weighted F1 0.8998 | macro AUROC 0.9854 | macro AUPRC 0.9087
Saved predictions and 3 result tables for run "logreg_context"

Per-class metrics (one class vs. the other four; decision rule: argmax):


,precision,sensitivity,specificity,f1,auroc,auprc,prevalence
class,,,,,,,
BACKGROUND,0.7519,0.8057,0.9657,0.7779,0.9788,0.8263,0.1142
OBJECTIVE,0.7376,0.6128,0.9814,0.6694,0.9751,0.7839,0.0786
METHODS,0.9404,0.9526,0.9703,0.9465,0.9897,0.9750,0.3298
RESULTS,0.9350,0.9410,0.9684,0.9380,0.9887,0.9791,0.3257
CONCLUSIONS,0.9318,0.9241,0.9879,0.9279,0.9948,0.9792,0.1517
macro average,0.8593,0.8472,0.9747,0.8519,0.9854,0.9087,0.2000



Confusion matrix (rows = actual class, columns = predicted class):


predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
actual,,,,,
BACKGROUND,2779,484,106,4,76
OBJECTIVE,823,1456,95,2,0
METHODS,55,28,9492,351,38
RESULTS,1,4,380,9260,196
CONCLUSIONS,38,2,21,287,4234


Classes never predicted: none


In [7]:
pipeline = best_text['model']
vocabulary = pipeline.named_steps['columntransformer'].named_transformers_['sentence'].get_feature_names_out()
weights = pipeline.named_steps['logisticregression'].coef_
model_classes = list(pipeline.classes_)
for c in CLASSES:
    top = np.argsort(weights[model_classes.index(c)])[-10:][::-1]
    print(f'{c:12s}: ' + ', '.join(vocabulary[top]))

BACKGROUND  : was to, is, has, are, have, aimed to, aimed, aim, was designed, recently
OBJECTIVE   : compare, whether, to, evaluate, to compare, investigate, to evaluate, to investigate, to test, examine
METHODS     : were, was, included, will, received, provides class, group, were compared, or, recorded
RESULTS     : were, was, 001, significantly, showed, 05, respectively, had, 01, 0001
CONCLUSIONS : should, improved, may, can, in, improves, further, similar, our, future
